In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# make columns viewed max =50
pd.set_option('display.max_columns', 50)

In [ ]:
df_raw = pd.read_csv("celonis_data.csv")


In [ ]:
columns_used = ['Id', 'NextReleaseDate', 'OldHorizonIssueDate', 'HorizonIssueDate',
       'ReleaseDate', 'DateTimeCreated', 'InternalPartNumber',
       'KeytoTRLP', 'ABCCode','Release', 'ReleaseQuantityNet', 'ReleaseQuantityCum',
       'OldRelease', 'OldQuantity', 'ShippedQ',]
df = df_raw[columns_used]

In [ ]:
df.loc[(df["DateTimeCreated"] != df["HorizonIssueDate"])]

In [ ]:
df.loc[df["InternalPartNumber"] == "92724A"].groupby("ReleaseDate").count().sort_values("Id", ascending=False).sort_values("ReleaseDate").tail(20)

In [ ]:
# check one part and make sense of dates

df_part = df[df["InternalPartNumber"] == "92724A"]
# select 5 random dates
dates = df_part["ReleaseDate"].drop_duplicates().sort_values().tail(10)
print(dates)
for date in dates:
    df_part_release = df_part.loc[df_part["ReleaseDate"] == date]
    print(f"Release Date: {date}")
    # Try to replicate the figure
    df_part_release.sort_values("DateTimeCreated").plot(
        x="DateTimeCreated", y="ReleaseQuantityNet", kind="line", marker = "o")
    NextReleaseDate = df_part_release["NextReleaseDate"].iloc[0]
    plt.title(f"Release Quantity Net over Time for Part 92724A on Release Date {date}, Next Release Date {NextReleaseDate}")
    plt.xlabel("Date Time Created")
    # rotate x ticks
    plt.xticks(rotation = 45)
    plt.ylabel("Release Quantity Net")

In [ ]:
df_part

In [ ]:
df_part["ReleaseDate"] = pd.to_datetime(df_part["ReleaseDate"])

In [ ]:
df_part_last = df_part.sort_values("DateTimeCreated").groupby("ReleaseDate").last().reset_index()
df_part_last = df_part_last.sort_values("ReleaseDate")
df_part_last["ReleaseDate"] = pd.to_datetime(df_part_last["ReleaseDate"])  # convert to datetime
fig, ax = plt.subplots(figsize=(14, 6))
data = df_part_last.iloc[-52:]
ax.bar(range(len(data)), data["ReleaseQuantityNet"], color="blue", width = 0.2)
ax.set_xticks(range(0, len(data), 2))  # show every 2nd tick
ax.set_xticklabels(data["ReleaseDate"].dt.strftime("%Y-%m-%d").iloc[::2], rotation=45, ha='right', )
ax.set_xlabel("Release Date")
ax.set_ylabel("Quantity")
plt.tight_layout()

In [ ]:
df[
    (df["InternalPartNumber"] == "92724A")
    & (df["ReleaseDate"] == "2025-12-22")
].sort_values(by="HorizonIssueDate")[["Release", "OldRelease", "InternalPartNumber", "ReleaseDate", "HorizonIssueDate"]]

In [ ]:
final_count = df_part_last.set_index("ReleaseDate")["ReleaseQuantityNet"].to_dict()
df_part["LeadTimeDays"] = (pd.to_datetime(df_part["ReleaseDate"]) - pd.to_datetime(df_part["DateTimeCreated"])).dt.days
df_part["final_count"] = df_part["ReleaseDate"].map(final_count)
sum_quantity = {}
for lead_time in [0, 7, 14, 21, 28, 45,  60]:
    df_part_lead_time = df_part[df_part["LeadTimeDays"] >lead_time]
    df_part_last = df_part_lead_time.sort_values("DateTimeCreated").groupby("ReleaseDate").last().reset_index()
    df_part_last = df_part_last.sort_values("ReleaseDate")
    # keep only the last 20 releases
    df_part_last = df_part_last.tail(20)
    plt.plot(df_part_last["ReleaseDate"], df_part_last["ReleaseQuantityNet"], marker = "o", label=f"Lead Time {lead_time} days")
    sum_quantity[lead_time] = df_part_last["ReleaseQuantityNet"].sum()
plt.xticks(rotation=45)
plt.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for release_date, group in df_part.sort_values("HorizonIssueDate").groupby("ReleaseDate"):
    if release_date < "2025-11-01": 
        continue # only plot for release dates before a certain date to avoid clutter
    ax.plot(group["HorizonIssueDate"], group["ReleaseQuantityNet"], marker="o", label=str(release_date)[:10])
ax.set_xlabel("Horizon Issue Date")
ax.set_ylabel("Release Quantity Net")
ax.set_title("Release Quantity Net over Horizon Issue Date (per Release Date)")
plt.xticks(rotation=45)
# Only show legend if not too many release dates
if df_part["ReleaseDate"].nunique() <= 20:
    ax.legend(title="Release Date", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.legend()

In [ ]:
final_quntity = sum_quantity[0]
keys = list(sum_quantity.keys())
values = list(sum_quantity.values())/final_quntity
plt.plot(keys, values, marker = "o")
for k, v in zip(keys, values):
    plt.text(k, v, f"{100*v:.0f}%", ha='center', va='bottom')
plt.xlabel("Lead Time (days)")
plt.ylabel("Percentage of Final Quantity")
plt.title("Percentage of Final Quantity vs Lead Time on average across releases for Part 92724A")
plt.gca().invert_xaxis()  # Invert x-axis to show lead time decreasing from left to right

In [ ]:
values

In [ ]:
df_part_agg = df_part.groupby("LeadTimeDays").agg(
    final_count=pd.NamedAgg(column="final_count", aggfunc="sum"),
    ReleaseQuantityNet=pd.NamedAgg(column="ReleaseQuantityNet", aggfunc="sum"),
).reset_index()
df_part_agg["percentagePlanned"] = df_part_agg["ReleaseQuantityNet"] / df_part_agg["final_count"] * 100
plt.plot(df_part_agg["LeadTimeDays"], df_part_agg["percentagePlanned"])
plt.xlabel("Lead Time (Days)")
plt.ylabel("Percentage Planned")
plt.title("Lead Time vs Percentage Planned for Part 92724A")
# inverse x-axis
plt.gca().invert_xaxis()

In [ ]:
df_part_agg

In [ ]:
df_part_release.sort_values("DateTimeCreated")

In [ ]:
df.groupby(["InternalPartNumber"]).agg(
    release_count=pd.NamedAgg(column="ReleaseDate", aggfunc="nunique"),
    release_max=pd.NamedAgg(column="ReleaseDate", aggfunc="max"),
    release_min=pd.NamedAgg(column="ReleaseDate", aggfunc="min"),
).sort_values("release_count", ascending=False)

In [ ]:
revision_counts = df.groupby(["InternalPartNumber", "ReleaseDate"]).agg(
    revision_count=pd.NamedAgg(column="DateTimeCreated", aggfunc="count"),
).sort_values("revision_count", ascending=False).reset_index()
parts = revision_counts["InternalPartNumber"].unique()
for part in parts[:10]:
    revision_counts_part = revision_counts[revision_counts["InternalPartNumber"] == part].sort_values("ReleaseDate")
    if len(revision_counts_part) > 1:
        plt.figure(figsize=(10, 6))
        plt.plot(revision_counts_part["ReleaseDate"], revision_counts_part["revision_count"], marker='o')
        plt.title(f"Revision Count over Release Dates for Part {part}")
        plt.xlabel("Release Date")
        plt.ylabel("Revision Count")
        plt.xticks(rotation=45)
        plt.grid()
        plt.show()

In [ ]:
revision_counts_part

In [ ]:
df.loc[df["Release"] == "240513-35045-0002"]

In [ ]:
df2 = pd.read_csv("celonis_data_forecast.csv")

In [ ]:
df2 = df2.dropna(subset = ["LightGBMForecast"])
df2.shape

In [ ]:
df2[["LightGBMForecast", "ProphetForecast", "ReleaseDate", "ReleaseQuantityNet", "InternalPartNumber"]].sample(6)

In [ ]:
df2.groupby(["ReleaseDate", "InternalPartNumber"]).count()

In [ ]:
df2.loc[(df2["ReleaseDate"] == "2026-01-05" ) &(df2["InternalPartNumber"] == "92507B"), ["", "", ]]

In [ ]:
df3 = df2.dropna(subset = ["ForecastQ", "LightGBMForecast", "ProphetForecast"])
df3.sample()[["ForecastQ", "LightGBMForecast", "ProphetForecast", "ReleaseDate", "InternalPartNumber", "ReleaseQuantityNet"]]

In [ ]:
df3_last = df3.sort_values("HorizonIssueDate").groupby(["ReleaseDate", "InternalPartNumber"]).last().reset_index()
df2_last = df2.sort_values("HorizonIssueDate").groupby(["ReleaseDate", "InternalPartNumber"]).last().reset_index()
df2_last.shape, df3_last.shape

In [ ]:
# calculate errors for ForecastQ", "LightGBMForecast", "ProphetForecast
df2_last["Error_LightGBM"] = df2_last["ReleaseQuantityNet"] - df2_last["LightGBMForecast"]
df2_last["Error_Prophet"] = df2_last["ReleaseQuantityNet"] - df2_last["ProphetForecast"]
df2_last["Error_LightGBM"].abs().sum()/df2_last["ReleaseQuantityNet"].abs().sum(), df2_last["Error_Prophet"].abs().sum()/df2_last["ReleaseQuantityNet"].abs().sum()

In [ ]:
pd.date_range(start ="2026-01-05", periods = 8, freq = "W-MON")

In [ ]:
df2_last["ReleaseDate"].unique()

In [ ]:
# calculate errors for ForecastQ", "LightGBMForecast", "ProphetForecast
df3_last["Error_ForecastQ"] = df3_last["ReleaseQuantityNet"] - df3_last["ForecastQ"]
df3_last["Error_LightGBM"] = df3_last["ReleaseQuantityNet"] - df3_last["LightGBMForecast"]
df3_last["Error_Prophet"] = df3_last["ReleaseQuantityNet"] - df3_last["ProphetForecast"]
df3_last["Error_ForecastQ"].abs().sum()/df3_last["ReleaseQuantityNet"].abs().sum(), df3_last["Error_LightGBM"].abs().sum()/df3_last["ReleaseQuantityNet"].abs().sum(), df3_last["Error_Prophet"].abs().sum()/df3_last["ReleaseQuantityNet"].abs().sum()

In [ ]:
df3_last.shape

In [ ]:
df3_last.loc[df3_last["Error_ForecastQ"] < 1].sample()[["ForecastQ", "LightGBMForecast", "ProphetForecast", "ReleaseDate", "InternalPartNumber", "ReleaseQuantityNet"]]

In [ ]:
df3_last.loc[df3_last["Error_ForecastQ"] < 1].shape

In [ ]:
df3_last.loc[df3_last["Error_ForecastQ"] > 1, ["ForecastQ", "LightGBMForecast", "ProphetForecast", "ReleaseDate", "InternalPartNumber", "ReleaseQuantityNet"]]

In [ ]:
df2.loc[(df2["InternalPartNumber"] == "92723A") & (df2["ReleaseDate"]  == "2026-02-23")]

In [ ]:
df2.loc[(df2["InternalPartNumber"] == "92723A") & (df2["ReleaseDate"]  == "2026-02-16")]

In [ ]:
# Ikigai SDK
from ikigai import Ikigai
# Standard library
import warnings

# Third-party
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import urllib3
import yaml
from utils import EXPLAINABILITY_METRICS_CONTEXT, app_info, upload_dataset, build_or_update_flow, run_flow



In [ ]:
# ── Credentials ──────────────────────────────────────────────────────────
with open("key_prod.yaml", "r") as file:
    keys = yaml.safe_load(file)

API_KEY = keys["API_KEY"]
BASE_URL = keys["BASE_URL"]
USER_EMAIL = keys["USER_EMAIL"]

headers = {"User": USER_EMAIL, "Api-key": API_KEY}
ikigai = Ikigai(USER_EMAIL, API_KEY, BASE_URL)
app_name = "Celonis PoC"
try:
    app = ikigai.apps[app_name]
except RuntimeError:
    app = ikigai.app.new(name=app_name).build()




In [ ]:
aux = app.datasets["aux"].df()

In [ ]:
aux

In [ ]:
# Pivot aux from long to wide: one column per lead signal
aux_wide = aux.pivot_table(
    index=["date", "InternalPartNumber", "ABCCode"],
    columns="identifier",
    values="value",
    aggfunc="sum",
).reset_index()

aux_wide = aux_wide.sort_values("date")
aux_wide = aux_wide.loc[aux_wide["date"]>= "2024-01-01"]
aux_wide = aux_wide.loc[aux_wide["date"]< "2026-01-01"]

# Reorder lead columns numerically
lead_cols = sorted(
    [c for c in aux_wide.columns if c.startswith("lead_")],
    key=lambda c: int(c.split("_")[1]),
)

# WMAPE of each lead_k signal vs lead_0 (the "actual" / latest-snapshot quantity)
actual = aux_wide["lead_0"]

wmape_results = {}
for col in lead_cols:
    # if col == "lead_0":
    #     continue
    predicted = aux_wide[col]
    mask = actual.abs() > 0  # only where there's a nonzero actual
    wmape_val = (actual[mask] - predicted[mask]).abs().sum() / actual[mask].abs().sum()
    wmape_results[col] = wmape_val

wmape_df = pd.DataFrame.from_dict(wmape_results, orient="index", columns=["WMAPE_vs_lead_0"])
wmape_df.index.name = "signal"
wmape_df = wmape_df.sort_index(key=lambda x: x.map(lambda s: int(s.split("_")[1])))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
days = [int(s.split("_")[1]) for s in wmape_df.index]
ax.bar(range(len(days)), wmape_df["WMAPE_vs_lead_0"], color="steelblue")
ax.set_xticks(range(len(days)))
ax.set_xticklabels([f"{d}d" for d in days])
for i, v in enumerate(wmape_df["WMAPE_vs_lead_0"]):
    ax.text(i, v + 0.005, f"{v:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Lead-time signal")
ax.set_ylabel("WMAPE vs lead_0 (actual)")
ax.set_title("How well does each lead-time signal predict the final quantity?")
plt.tight_layout()
plt.show()

wmape_df

In [ ]:
# Per-part WMAPE for each lead signal vs lead_0
actual_col = "lead_0"
rows = []
for part, grp in aux_wide.groupby("InternalPartNumber"):
    act = grp[actual_col]
    denom = act.abs().sum()
    if denom == 0:
        continue
    for col in lead_cols:
        if col == "lead_0":
            continue
        wmape_val = (act - grp[col]).abs().sum() / denom
        rows.append({"InternalPartNumber": part, "signal": col, "wmape": wmape_val})

wmape_per_part = pd.DataFrame(rows)
wmape_per_part["lead_days"] = wmape_per_part["signal"].str.split("_").str[1].astype(int)

# Box plot — distribution of per-part WMAPE across lead signals
fig, ax = plt.subplots(figsize=(12, 6))
signals_sorted = sorted(wmape_per_part["signal"].unique(), key=lambda s: int(s.split("_")[1]))
data_to_plot = [wmape_per_part.loc[wmape_per_part["signal"] == s, "wmape"].values for s in signals_sorted]
bp = ax.boxplot(data_to_plot, labels=[f"{int(s.split('_')[1])}d" for s in signals_sorted],
                patch_artist=True, showfliers=False, medianprops=dict(color="black"))
for patch in bp["boxes"]:
    patch.set_facecolor("steelblue")
    patch.set_alpha(0.7)

# Overlay the global WMAPE as red dots
global_vals = [wmape_df.loc[s, "WMAPE_vs_lead_0"] for s in signals_sorted]
ax.scatter(range(1, len(signals_sorted) + 1), global_vals, color="red", zorder=5, s=60, label="Global WMAPE")

ax.set_xlabel("Lead-time signal")
ax.set_ylabel("WMAPE vs lead_0 (actual)")
ax.set_title("Distribution of per-part WMAPE across lead-time signals\n(boxes = part-level, red dots = global)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-date WMAPE for each lead signal vs lead_0, one subplot per lead
signals_sorted = sorted(
    [c for c in lead_cols if c != "lead_0"],
    key=lambda s: int(s.split("_")[1]),
)

n = len(signals_sorted)
fig, axes = plt.subplots(n, 1, figsize=(14, 3.5 * n), sharex=True, sharey=True)

for i, sig in enumerate(signals_sorted):
    ax = axes[i]
    days_label = int(sig.split("_")[1])

    # Compute WMAPE per date
    per_date = []
    for dt, grp in aux_wide.groupby("date"):
        act = grp["lead_0"]
        denom = act.abs().sum()
        if denom == 0:
            continue
        w = (act - grp[sig]).abs().sum() / denom
        per_date.append({"date": dt, "wmape": w})

    pdf = pd.DataFrame(per_date).sort_values("date")
    pdf["date"] = pd.to_datetime(pdf["date"])

    ax.bar(pdf["date"], pdf["wmape"], width=5, color="steelblue", alpha=0.7)
    # Rolling average
    if len(pdf) >= 4:
        ax.plot(pdf["date"], pdf["wmape"].rolling(4, center=True).mean(),
                color="red", linewidth=2, label="4-week MA")
    ax.axhline(wmape_df.loc[sig, "WMAPE_vs_lead_0"], color="black",
               linestyle="--", linewidth=1, label=f"Global: {wmape_df.loc[sig, 'WMAPE_vs_lead_0']:.1%}")
    ax.set_ylabel("WMAPE")
    ax.set_title(f"lead_{days_label}  ({days_label}-day signal)")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_ylim(0, 1.5)

axes[-1].set_xlabel("Week")
fig.suptitle("Weekly WMAPE of each lead signal vs actual (lead_0)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()